
#### 2.7 Experiments: Problem (tokenizer_experiments): Experiments with tokenizers


In [76]:
vocab_tn = "results/tiny_stories/tiny_stories_vocab.pkl"
merges_tn = "results/tiny_stories/tiny_stories_merges.pkl"

vocab_owt = "results/owt/owt_train_vocab.pkl"
merges_owt = "results/owt/owt_train_merge.pkl"

special_tokens = ["<|endoftext|>"]

tn_filepath = "data/TinyStoriesV2-GPT4-train.txt"
owt_filepath = "data/owt_train.txt"


In [74]:
import pickle

def get_files(vocab_file, merge_file):
    with open(vocab_file, "rb") as f:
        vocab = pickle.load(f)
    with open(merge_file, "rb") as f:
        merges= pickle.load(f)
    return vocab, merges

In [75]:
from cs336_basics.tokenizer import Tokenizer

tokenizer_owt = Tokenizer.from_files(vocab_owt, merges_owt, special_tokens)
tokenizer_tn = Tokenizer.from_files(vocab_tn, merges_tn, special_tokens)

#### (a) Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively), encode these sampled documents into integer IDs. What is each tokenizer’s compression ratio (bytes/token)?
#### b) What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer? Compare the compression ratio and/or qualitatively describe what happens.

In [78]:
from typing import Iterator

class DocumentSampler:
    def __init__(self, file_path: str, delimiter: str = "<|endoftext|>", chunk_size: int = 1) -> None:
        self.file_path = file_path
        self.delimiter = delimiter
        self.chunk_size = chunk_size * 1024 *1024 # MB
    
    def sample_all(self) -> Iterator[str]:
        with open(self.file_path, "r", encoding="utf-8", errors="replace") as file:
            while True:
                chunk = file.read(self.chunk_size)
                if chunk == "":
                    break
                yield chunk
    
    def sample(self, num_samples: int = 10):
        buffer = ""
        chunk = ""
        docs_sampled = 0
        with open(self.file_path, "r", encoding="utf-8", errors="replace") as file:
            while docs_sampled < num_samples:
                chunk = file.read(self.chunk_size)
                if chunk == "":
                    break
                chunk = buffer + chunk
                parts = chunk.split(self.delimiter)

                buffer = parts[-1]
                docs = parts[:-1]

                for doc in docs:
                    if doc:
                        yield doc
                        docs_sampled += 1
                    if docs_sampled >= num_samples:
                        return

In [79]:
owt_sampler = DocumentSampler(owt_filepath)
tn_sampler = DocumentSampler(tn_filepath)

In [84]:
def get_compression_ratio(documents: list[str], tokenizer) -> dict:
    """Calculate compression metrics for a list of documents"""
    
    total_bytes = 0
    total_tokens = 0
    ratios = []
    
    for doc in documents:
        # Calculate bytes (UTF-8 encoding)
        doc_bytes = len(doc.encode('utf-8'))
        
        # Tokenize
        token_ids = tokenizer.encode(doc)
        doc_tokens = len(token_ids)
        
        # Individual document ratio
        doc_ratio = doc_bytes / doc_tokens if doc_tokens > 0 else 0
        ratios.append(doc_ratio)
        
        # Accumulate totals
        total_bytes += doc_bytes
        total_tokens += doc_tokens
    
    # Overall compression ratio
    overall_ratio = total_bytes / total_tokens if total_tokens > 0 else 0
    
    return {
        'overall_ratio': overall_ratio,
        'individual_ratios': ratios,
        'avg_ratio': sum(ratios) / len(ratios) if ratios else 0,
        'total_bytes': total_bytes,
        'total_tokens': total_tokens
    }

# Usage for your experiment:
def analyze_compression():
    # Sample documents
    owt_docs = list(owt_sampler.sample(20))
    tn_docs = list(tn_sampler.sample(20))
    
    print("=== OpenWebText Documents ===")
    
    # Test OWT tokenizer on OWT data
    owt_on_owt = get_compression_ratio(owt_docs, tokenizer_owt)
    print(f"OWT tokenizer on OWT data: {owt_on_owt['overall_ratio']:.2f} bytes/token")
    
    # Test TN tokenizer on OWT data  
    tn_on_owt = get_compression_ratio(owt_docs, tokenizer_tn)
    print(f"TN tokenizer on OWT data: {tn_on_owt['overall_ratio']:.2f} bytes/token")
    
    print("\n=== TinyStories Documents ===")
    
    # Test OWT tokenizer on TN data
    owt_on_tn = get_compression_ratio(tn_docs, tokenizer_owt)
    print(f"OWT tokenizer on TN data: {owt_on_tn['overall_ratio']:.2f} bytes/token")
    
    # Test TN tokenizer on TN data
    tn_on_tn = get_compression_ratio(tn_docs, tokenizer_tn)
    print(f"TN tokenizer on TN data: {tn_on_tn['overall_ratio']:.2f} bytes/token")
    
    return {
        'owt_on_owt': owt_on_owt,
        'tn_on_owt': tn_on_owt,
        'owt_on_tn': owt_on_tn,
        'tn_on_tn': tn_on_tn
    }

results = analyze_compression()

=== OpenWebText Documents ===
OWT tokenizer on OWT data: 4.46 bytes/token
TN tokenizer on OWT data: 3.15 bytes/token

=== TinyStories Documents ===
OWT tokenizer on TN data: 3.97 bytes/token
TN tokenizer on TN data: 4.06 bytes/token


##### They both encode about 4 bytes per token for their respective datasets. 
##### The tinystories tokenizer drops in performance in compression ratio when used on the Openwebtext dataset. This intuitively makes sense, because there are more unique tokens it hasn't probably seen in this dataset.